In [29]:
import random
import simpy
import pandas as pd

# Cria environment
random.seed(1000)
env = simpy.Environment()

# Criação dos recursos
auxiliar_adm = simpy.Resource(env, capacity=9)
medico = simpy.Resource(env, capacity=27)
equipe_enfermagem = simpy.Resource(env, capacity=16)

# Dicionário para armazenar métricas de cada exame
metricas_exames = {}

# Variáveis para rastreamento de número de exames no sistema
num_exames_em_fila = 0
num_exames_em_processamento = 0
num_exames_no_sistema = 0

# Lista para armazenar snapshots do sistema ao longo do tempo
snapshots_sistema = []

def distribuicoes(tipo):
    return {
        # Tempos de processo
        'dist_organizar_pedido': random.triangular(15, 40, 30),
        'dist_avaliar_prioridade': random.triangular(10, 15, 10),
        'dist_agendar_exame': random.triangular(30, 40, 30),
        'dist_informar_pre_requisito': random.triangular(5, 30, 20),
        'dist_preparar_paciente': random.triangular(5, 30, 20),
        'dist_transportar_paciente': random.triangular(5, 30, 20),
        'dist_avaliacao_de_realizacao_de_exame': random.triangular(5, 30, 20),
        'dist_avaliacao_exame_pode_ser_no_hospital': random.triangular(5, 30, 20),
        'dist_realizacao_do_exame_fora_do_hospital': random.triangular(7733, 11840, 8400),
        'dist_realizacao_do_exame_no_hospital': random.triangular(60, 720, 120),
        'dist_retorno_paciente': random.triangular(5, 30, 20),
        'dist_atualizar_mv': random.triangular(1, 5, 3),
        'dist_analisar_exame_urgente': random.triangular(1, 2, 1.5),
        'dist_analisar_exame_nao_urgente': random.triangular(1, 12, 6)
    }.get(tipo, 0.0)

# Porcentagens de decisão
x = 0.7  # Porcentagem dos exames que são urgentes
y = 0.005  # Porcentagem de pacientes que precisam ser reagendados
z = 0.0049  # Porcentagem de exames que devem ser feitos fora do hospital

# Outros parâmetros
taxa_de_chegada_exames = 0.295
limite_preparacao_exame = 3

def inicializar_metricas_exame(exame):
    """Inicializa o dicionário de métricas para um exame"""
    if exame not in metricas_exames:
        metricas_exames[exame] = {
            'exame': exame,
            'atividades_tempo_fila': {},
            'atividades_tempo_processamento': {}
        }

def registrar_snapshot_sistema(env):
    """Registra o estado do sistema no momento atual"""
    snapshots_sistema.append({
        'tempo': env.now,
        'num_em_fila': num_exames_em_fila,
        'num_em_processamento': num_exames_em_processamento,
        'num_no_sistema': num_exames_no_sistema
    })

def registrar_metrica(exame, nome_atividade, tempo_fila, tempo_processamento):
    """Registra tempo de fila e processamento de uma atividade"""
    if nome_atividade not in metricas_exames[exame]['atividades_tempo_fila']:
        metricas_exames[exame]['atividades_tempo_fila'][nome_atividade] = 0
        metricas_exames[exame]['atividades_tempo_processamento'][nome_atividade] = 0
    
    metricas_exames[exame]['atividades_tempo_fila'][nome_atividade] += tempo_fila
    metricas_exames[exame]['atividades_tempo_processamento'][nome_atividade] += tempo_processamento

def AtividadeCom1RecursoExame(env, exame, nome_da_atividade, recurso, tempo_atividade):
    global num_exames_em_fila, num_exames_em_processamento
    
    print("[%.2f] Exame %s chega à atividade '%s'" % (env.now, exame, nome_da_atividade))
    
    # Entra na fila
    num_exames_em_fila += 1
    registrar_snapshot_sistema(env)
    
    requisicao = recurso.request()
    tempo_inicio_fila = env.now
    yield requisicao
    tempo_em_fila = env.now - tempo_inicio_fila
    
    # Sai da fila e entra em processamento
    num_exames_em_fila -= 1
    num_exames_em_processamento += 1
    registrar_snapshot_sistema(env)
    
    print("[%.2f] Exame %s inicia a atividade '%s' (Tempo em fila = %.2f)" % 
          (env.now, exame, nome_da_atividade, tempo_em_fila))
    
    tempo_inicio_processamento = env.now
    yield env.timeout(tempo_atividade)
    tempo_processamento = env.now - tempo_inicio_processamento
    
    # Sai do processamento
    num_exames_em_processamento -= 1
    registrar_snapshot_sistema(env)
    
    yield recurso.release(requisicao)
    print("[%.2f] Exame %s na atividade '%s' finalizado" % (env.now, exame, nome_da_atividade))
    
    # Registra métricas
    registrar_metrica(exame, nome_da_atividade, tempo_em_fila, tempo_processamento)

def AtividadeCom1RecursoPaciente(env, exame, nome_da_atividade, recurso, tempo_atividade):
    global num_exames_em_fila, num_exames_em_processamento
    
    print("[%.2f] Paciente do exame %s chega à atividade '%s'" % (env.now, exame, nome_da_atividade))
    
    # Entra na fila
    num_exames_em_fila += 1
    registrar_snapshot_sistema(env)
    
    requisicao = recurso.request()
    tempo_inicio_fila = env.now
    yield requisicao
    tempo_em_fila = env.now - tempo_inicio_fila
    
    # Sai da fila e entra em processamento
    num_exames_em_fila -= 1
    num_exames_em_processamento += 1
    registrar_snapshot_sistema(env)
    
    print("[%.2f ] Paciente do exame %s inicia a atividade '%s'(Tempo em fila = %.2f)" % 
          (env.now, exame, nome_da_atividade, tempo_em_fila))
    
    tempo_inicio_processamento = env.now
    yield env.timeout(tempo_atividade)
    tempo_processamento = env.now - tempo_inicio_processamento
    
    # Sai do processamento
    num_exames_em_processamento -= 1
    registrar_snapshot_sistema(env)
    
    yield recurso.release(requisicao)
    print("[%.2f] Paciente do exame %s na atividade '%s' finalizado" % 
          (env.now, exame, nome_da_atividade))
    
    # Registra métricas
    registrar_metrica(exame, nome_da_atividade, tempo_em_fila, tempo_processamento)

def AtividadeSemRecurso(env, exame, nome_da_atividade, tempo_atividade):
    """Para atividades que consomem tempo mas não usam recursos"""
    global num_exames_em_processamento
    
    print("[%.2f] Exame %s chega à atividade '%s'" % (env.now, exame, nome_da_atividade))
    print("[%.2f] Exame %s inicia a atividade '%s'" % (env.now, exame, nome_da_atividade))
    
    # Entra em processamento
    num_exames_em_processamento += 1
    registrar_snapshot_sistema(env)
    
    tempo_inicio_processamento = env.now
    yield env.timeout(tempo_atividade)
    tempo_processamento = env.now - tempo_inicio_processamento
    
    # Sai do processamento
    num_exames_em_processamento -= 1
    registrar_snapshot_sistema(env)
    
    print("[%.2f] Exame %s na atividade '%s' finalizado" % (env.now, exame, nome_da_atividade))
    
    # Registra métricas (sem fila para atividades sem recurso)
    registrar_metrica(exame, nome_da_atividade, 0, tempo_processamento)

def geraChegadasExames(env, taxa_exames):
    global num_exames_no_sistema
    contaChegadaExame = 0
    while True:
        yield env.timeout(random.expovariate(1/taxa_exames))
        contaChegadaExame += 1
        print("[%.2f] Exame %s chega ao sistema" % (env.now, contaChegadaExame))
        
        # Exame entra no sistema
        num_exames_no_sistema += 1
        registrar_snapshot_sistema(env)
        
        # Inicializa métricas do exame
        inicializar_metricas_exame(contaChegadaExame)
        
        env.process(procExames(env, contaChegadaExame, auxiliar_adm, medico, equipe_enfermagem))

def procExames(env, exame, auxiliar_adm, medico, equipe_enfermagem):
    global num_exames_no_sistema
    
    # Atributos do exame
    Prob_Urgencia = random.uniform(0, 1)
    if Prob_Urgencia <= x:
        A_urgencia = 1
    else:
        A_urgencia = 0
    
    instante_inicio_processo = env.now
    
    # Atividade 01 - Organizar pedidos
    nome_da_atividade = "01 - Organizar pedidos"
    yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, auxiliar_adm, 
                                         distribuicoes('dist_organizar_pedido'))
    
    # Atividade 02 - Avaliar prioridade
    nome_da_atividade = "02 - Avaliar prioridade"
    yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, medico, 
                                         distribuicoes('dist_avaliar_prioridade'))
    
    if A_urgencia == 1:
        print("Exame urgente!!!")
        yield from procPreRealizacaoExames(env, exame, equipe_enfermagem)
    else:
        print("Exame não urgente")
        nome_da_atividade = "3 - Agendar exame"
        yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, auxiliar_adm, 
                                             distribuicoes('dist_agendar_exame'))
        yield from procPreRealizacaoExames(env, exame, equipe_enfermagem)
    
    # Sorteia se o paciente está preparado
    if random.random() > y:
        paciente_preparado = 1
    else:
        paciente_preparado = 0
    
    passagem_no_loop = 0
    while paciente_preparado == 0 and passagem_no_loop < limite_preparacao_exame:
        nome_da_atividade = "03 - Agendar exame"
        print("Reagendamento do exame %s por paciente não estar preparado" % (exame))
        yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, auxiliar_adm, 
                                             distribuicoes('dist_agendar_exame'))
        yield from procPreRealizacaoExames(env, exame, equipe_enfermagem)
        passagem_no_loop += 1
        
        if random.random() > y:
            paciente_preparado = 1
        else:
            paciente_preparado = 0
    
    # Atividade 8 - Avaliar se exame pode ser feito no hospital
    nome_da_atividade = "08 - Avaliar se exame pode ser feito no hospital"
    yield from AtividadeSemRecurso(env, exame, nome_da_atividade, 
                                   distribuicoes('dist_avaliacao_exame_pode_ser_no_hospital'))
    
    # Sorteia se exame pode ou não ser feito no hospital
    if random.random() <= z:
        exame_fora_do_hospital = 1
    else:
        exame_fora_do_hospital = 0
    
    if exame_fora_do_hospital == 1:
        nome_da_atividade = "09 - Realizar exame fora do hospital"
        yield from AtividadeSemRecurso(env, exame, nome_da_atividade, 
                                       distribuicoes('dist_realizacao_do_exame_fora_do_hospital'))
    else:
        nome_da_atividade = "10 - Realizar exame no hospital"
        yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, equipe_enfermagem, 
                                             distribuicoes('dist_realizacao_do_exame_no_hospital'))
        env.process(procRetornoPaciente(env, exame, equipe_enfermagem))
    
    # Atividade 11 - Atualizar MV
    nome_da_atividade = "11 - Atualizar MV"
    yield from AtividadeSemRecurso(env, exame, nome_da_atividade, 
                                   distribuicoes('dist_atualizar_mv'))
    
    if A_urgencia == 1:
        nome_da_atividade = "13 - Analisar exame urgente"
        yield from AtividadeSemRecurso(env, exame, nome_da_atividade, 
                                       distribuicoes('dist_analisar_exame_urgente'))
    else:
        nome_da_atividade = "14 - Analisar exame não urgente"
        yield from AtividadeSemRecurso(env, exame, nome_da_atividade, 
                                       distribuicoes('dist_analisar_exame_nao_urgente'))
    
    # Exame sai do sistema
    num_exames_no_sistema -= 1
    registrar_snapshot_sistema(env)
    
    instante_final_do_processo = env.now
    duracao_do_processo = instante_final_do_processo - instante_inicio_processo
    print("[%.2f] Fim do processo do exame %s, no instante %.2f. Duração total = %.2f" % 
          (env.now, exame, instante_final_do_processo, duracao_do_processo))

def procPreRealizacaoExames(env, exame, equipe_enfermagem):
    # Atividade 4 - Informar pré-requisitos
    nome_da_atividade = "04 - Informar pré-requisitos"
    yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, equipe_enfermagem, 
                                         distribuicoes('dist_informar_pre_requisito'))
    
    # Atividade 5 - Preparar Paciente
    nome_da_atividade = "05 - Preparar paciente"
    yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, equipe_enfermagem, 
                                         distribuicoes('dist_preparar_paciente'))
    
    # Atividade 6 - Transportar paciente
    nome_da_atividade = "06 - Transportar paciente"
    yield from AtividadeCom1RecursoExame(env, exame, nome_da_atividade, equipe_enfermagem, 
                                         distribuicoes('dist_transportar_paciente'))
    
    # Atividade 7 - Avaliação de realização de exame
    nome_da_atividade = "07 - Avaliação de realização de exame"
    yield from AtividadeSemRecurso(env, exame, nome_da_atividade, 
                                   distribuicoes('dist_avaliacao_de_realizacao_de_exame'))

def procRetornoPaciente(env, exame, equipe_enfermagem):
    # Atividade 12 - Retorno do paciente ao PS
    nome_da_atividade = "12 - Retorno do paciente ao PS"
    yield from AtividadeCom1RecursoPaciente(env, exame, nome_da_atividade, equipe_enfermagem, 
                                            distribuicoes('dist_retorno_paciente'))

def gerar_dataframe_exames():
    """Gera o dataframe com todas as métricas coletadas"""
    dados = []
    
    for exame_id, metricas in metricas_exames.items():
        linha = {'Exame': exame_id}
        
        # Adiciona colunas de tempo em fila para cada atividade
        for atividade, tempo in metricas['atividades_tempo_fila'].items():
            linha[f"{atividade} - TempoEmFila"] = tempo
        
        # Adiciona colunas de tempo em processamento para cada atividade
        for atividade, tempo in metricas['atividades_tempo_processamento'].items():
            linha[f"{atividade} - TempoEmProcessamento"] = tempo
        
        # Calcula tempo total em fila
        linha['Tempo total em fila'] = sum(metricas['atividades_tempo_fila'].values())
        
        # Calcula tempo total em processamento
        linha['Tempo total em processamento'] = sum(metricas['atividades_tempo_processamento'].values())
        
        # Calcula tempo no sistema
        linha['TempoNoSistema'] = linha['Tempo total em fila'] + linha['Tempo total em processamento']
        
        dados.append(linha)
    
    df = pd.DataFrame(dados)
    
    # Reordena colunas: Exame primeiro, depois atividades, depois totais
    colunas_ordenadas = ['Exame']
    colunas_atividades = [col for col in df.columns if col not in ['Exame', 'Tempo total em fila', 'Tempo total em processamento', 'TempoNoSistema']]
    colunas_ordenadas.extend(sorted(colunas_atividades))
    colunas_ordenadas.extend(['Tempo total em fila', 'Tempo total em processamento', 'TempoNoSistema'])
    
    df = df[colunas_ordenadas]
    df = df.fillna(0)  # Preenche NaN com 0 para atividades não realizadas
    
    return df

def calcular_numero_medio_exames():
    """Calcula o número médio de exames em fila, processamento e no sistema"""
    if len(snapshots_sistema) < 2:
        return 0, 0, 0

    # Calcula a média ponderada pelo tempo
    tempo_total = snapshots_sistema[-1]['tempo'] - snapshots_sistema[0]['tempo']
    
    soma_ponderada_fila = 0
    soma_ponderada_processamento = 0
    soma_ponderada_sistema = 0
    
    for i in range(len(snapshots_sistema) - 1):
        intervalo_tempo = snapshots_sistema[i+1]['tempo'] - snapshots_sistema[i]['tempo']
        
        soma_ponderada_fila += snapshots_sistema[i]['num_em_fila'] * intervalo_tempo
        soma_ponderada_processamento += snapshots_sistema[i]['num_em_processamento'] * intervalo_tempo
        soma_ponderada_sistema += snapshots_sistema[i]['num_no_sistema'] * intervalo_tempo
    
    num_medio_em_fila = soma_ponderada_fila / tempo_total if tempo_total > 0 else 0
    num_medio_em_processamento = soma_ponderada_processamento / tempo_total if tempo_total > 0 else 0
    num_medio_no_sistema = soma_ponderada_sistema / tempo_total if tempo_total > 0 else 0
    
    return num_medio_em_fila, num_medio_em_processamento, num_medio_no_sistema

# Execução da simulação
env.process(geraChegadasExames(env, taxa_de_chegada_exames))
tempo_de_rodada = 10000
env.run(until=tempo_de_rodada)



[0.44] Exame 1 chega ao sistema
[0.44] Exame 1 chega à atividade '01 - Organizar pedidos'
[0.44] Exame 1 inicia a atividade '01 - Organizar pedidos' (Tempo em fila = 0.00)
[0.77] Exame 2 chega ao sistema
[0.77] Exame 2 chega à atividade '01 - Organizar pedidos'
[0.77] Exame 2 inicia a atividade '01 - Organizar pedidos' (Tempo em fila = 0.00)
[0.97] Exame 3 chega ao sistema
[0.97] Exame 3 chega à atividade '01 - Organizar pedidos'
[0.97] Exame 3 inicia a atividade '01 - Organizar pedidos' (Tempo em fila = 0.00)
[1.08] Exame 4 chega ao sistema
[1.08] Exame 4 chega à atividade '01 - Organizar pedidos'
[1.08] Exame 4 inicia a atividade '01 - Organizar pedidos' (Tempo em fila = 0.00)
[1.27] Exame 5 chega ao sistema
[1.27] Exame 5 chega à atividade '01 - Organizar pedidos'
[1.27] Exame 5 inicia a atividade '01 - Organizar pedidos' (Tempo em fila = 0.00)
[1.39] Exame 6 chega ao sistema
[1.39] Exame 6 chega à atividade '01 - Organizar pedidos'
[1.39] Exame 6 inicia a atividade '01 - Organizar 

In [30]:
# Gera e exibe o dataframe de métricas
df_exames = gerar_dataframe_exames()
#print("\n" + "="*100)
#print("DATAFRAME DE MÉTRICAS DOS EXAMES")
#print("="*100)
#print(df_exames.to_string())

print("snapshots_sistema")
#print(snapshots_sistema)
df_snapshots_sistema = pd.DataFrame(snapshots_sistema)
# Salva em CSV - Snapshots para número de exames
df_snapshots_sistema.to_csv('snapshot_sistemas.csv', index=False)
print("\n\nDataframe salvo em 'snapshot_sistemas.csv'")

# Salva em CSV - Métricas para cálculos de tempos
df_exames.to_csv('metricas_exames.csv', index=False)
print("\n\nDataframe salvo em 'metricas_exames.csv'")

# Coleta das estatísticas baseadas em tempo
total_exames_simulados = len(df_exames)
tempo_medio_em_fila = df_exames['Tempo total em fila'].mean()
tempo_medio_em_processamento = df_exames['Tempo total em processamento'].mean()
tempo_medio_no_sistema = df_exames['TempoNoSistema'].mean()

# Coleta das estatísticas baseadas em número de exames
num_medio_exames_em_fila, num_medio_exames_em_processamento, num_medio_exames_no_sistema = calcular_numero_medio_exames()

# Estatísticas resumidas
print("\n" + "="*100)
print("ESTATÍSTICAS RESUMIDAS")
print("="*100)
print("\n--- Estatísticas baseadas em TEMPO ---")
print(f"1) Total de exames simulados: {total_exames_simulados}")
print(f"2) Tempo médio dos exames em fila: {tempo_medio_em_fila:.2f}")
print(f"3) Tempo médio dos exames em processamento: {tempo_medio_em_processamento:.2f}")
print(f"4) Tempo médio dos exames no sistema: {tempo_medio_no_sistema:.2f}")
print("\n--- Estatísticas baseadas em NÚMERO DE EXAMES ---")
print(f"5) Número médio de exames em fila: {num_medio_exames_em_fila:.2f}")
print(f"6) Número médio de exames em processamento: {num_medio_exames_em_processamento:.2f}")
print(f"7) Número médio de exames no sistema: {num_medio_exames_no_sistema:.2f}")

snapshots_sistema


Dataframe salvo em 'snapshot_sistemas.csv'


Dataframe salvo em 'metricas_exames.csv'

ESTATÍSTICAS RESUMIDAS

--- Estatísticas baseadas em TEMPO ---
1) Total de exames simulados: 33824
2) Tempo médio dos exames em fila: 627.64
3) Tempo médio dos exames em processamento: 9.06
4) Tempo médio dos exames no sistema: 636.70

--- Estatísticas baseadas em NÚMERO DE EXAMES ---
5) Número médio de exames em fila: 16728.57
6) Número médio de exames em processamento: 30.83
7) Número médio de exames no sistema: 16704.03
